In [22]:
import numpy as np
import open3d as o3d
from scipy.spatial import KDTree
import matplotlib as plt

In [23]:
def load_ply_file(ply_file_path, voxel_size=0.01):
    """Load PLY file using Open3D and extract point cloud data."""
    pcd = o3d.io.read_point_cloud(ply_file_path)
    
    # Estimate normals using provided method
    radius_normal = voxel_size * 5
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius_normal, max_nn=50
        )
    )
    pcd.orient_normals_consistent_tangent_plane(k=10)
    pcd.normals = o3d.utility.Vector3dVector(-np.asarray(pcd.normals))
    
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)
    normals = np.asarray(pcd.normals)
    
    curvatures = estimate_curvature(pcd)  # Compute curvatures using neighboring points
    return pcd, points, colors, normals, curvatures


In [24]:
def estimate_curvature(pcd, voxel_size=0.01):
    """Estimate curvature based on variance of normals in the neighborhood."""
    radius_normal = voxel_size * 5
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=50))
    
    # Create KDTree for fast nearest-neighbor search
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)

    # Calculate curvature based on variance of normals
    curvatures = []
    for i in range(len(pcd.points)):
        k, idx, _ = pcd_tree.search_radius_vector_3d(pcd.points[i], radius_normal)
        if k > 1:
            neighbor_normals = np.asarray(pcd.normals)[idx, :]
            curvature = np.mean(np.linalg.norm(np.cross(neighbor_normals - pcd.normals[i], neighbor_normals), axis=1))
            curvatures.append(curvature)
        else:
            curvatures.append(0)

    return np.array(curvatures)

def visualize_curvature_colormap(pcd, curvatures):
    """Visualize curvature by coloring the point cloud according to curvature values."""
    curvature_colors = plt.cm.viridis(curvatures / np.max(curvatures))[:, :3]  # Normalize curvature for colormap
    pcd.colors = o3d.utility.Vector3dVector(curvature_colors)
    
    o3d.visualization.draw_geometries([pcd], window_name="Curvature Colormap Visualization")


In [25]:
def compute_superpoint_features(points, colors, normals, curvatures):
    """Compute features for each superpoint identified by its color."""
    superpoint_features = {}
    unique_colors = np.unique(colors, axis=0)
    
    for color in unique_colors:
        sp_mask = np.all(colors == color, axis=1)
        sp_points = points[sp_mask]
        sp_normals = normals[sp_mask]
        sp_curvatures = curvatures[sp_mask]
        centroid = np.mean(sp_points, axis=0)
        avg_normal = np.mean(sp_normals, axis=0)
        avg_curvature = np.mean(sp_curvatures)
        
        superpoint_features[tuple(color)] = {
            'centroid': centroid,
            'avg_normal': avg_normal,
            'avg_curvature': avg_curvature,
            'points': sp_points,
            'indices': np.where(sp_mask)[0]  # Save indices of the points for later coloring
        }
    
    return superpoint_features

In [26]:
def find_adjacent_superpoints(superpoint_features, distance_threshold=0.05):
    """Determine which superpoints are adjacent using KD-Tree based on their centroids."""
    centroids = np.array([features['centroid'] for features in superpoint_features.values()])
    colors = list(superpoint_features.keys())
    
    kd_tree = KDTree(centroids)
    adjacency_list = {color: [] for color in colors}
    
    for idx, color in enumerate(colors):
        # Query the k nearest neighbors (including the superpoint itself)
        distances, indices = kd_tree.query(centroids[idx], k=min(10, len(centroids)))  # Adjust k based on available neighbors
        for j, neighbor_idx in enumerate(indices):
            if neighbor_idx != idx and distances[j] < distance_threshold:
                neighbor_color = colors[neighbor_idx]
                adjacency_list[color].append(neighbor_color)
    
    return adjacency_list


In [49]:
def find_seed_superpoint(points, colors):
    """Find the seed superpoint based on the highest point closest to the centroid."""
    min_bound = points.min(axis=0)
    max_bound = points.max(axis=0)

    centroid_x = (min_bound[0] + max_bound[0]) / 2.0
    centroid_y = (min_bound[1] + max_bound[1]) / 2.0

    distances = np.linalg.norm(points[:, :2] - np.array([centroid_x, centroid_y]), axis=1)
    highest_point_index = np.argmax(points[:, 2] - distances)
    
    # Get the color of the seed point
    rock_seed_color = tuple(colors[highest_point_index])
    
    return rock_seed_color

In [28]:
def precompute_neighbors(pcd_tree, centroids, distance_threshold):
    """Precompute the neighbors for each point in the point cloud based on a distance threshold."""
    neighbors = []
    for i in range(len(centroids)):
        idx = pcd_tree.query_ball_point(centroids[i], distance_threshold)
        neighbors.append(np.array(idx[1:]))  # Skip the first index as it's the point itself
    return neighbors

In [29]:
def visualize_superpoint_normals(pcd, superpoint_features):
    """Visualize the average normals of superpoints by coloring them accordingly."""
    colors = np.asarray(pcd.colors)
    
    # Normalize the average normals for visualization
    normalized_normals = np.array([features['avg_normal'] for features in superpoint_features.values()])
    normalized_normals -= normalized_normals.min(axis=0)
    normalized_normals /= normalized_normals.max(axis=0)
    
    # Assign colors based on the normalized average normals
    for i, color in enumerate(superpoint_features.keys()):
        indices = superpoint_features[color]['indices']
        colors[indices] = normalized_normals[i]
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # Visualize the point cloud colored by superpoint normals
    o3d.visualization.draw_geometries([pcd], window_name="Superpoint Normals Visualization")

def visualize_superpoint_curvature(pcd, superpoint_features):
    """Visualize the average curvature of superpoints by coloring them accordingly."""
    colors = np.asarray(pcd.colors)
    
    # Normalize the average curvatures for visualization
    avg_curvatures = np.array([features['avg_curvature'] for features in superpoint_features.values()])
    normalized_curvatures = (avg_curvatures - avg_curvatures.min()) / (avg_curvatures.max() - avg_curvatures.min())
    
    # Map curvatures to a colormap
    curvature_colors = plt.cm.viridis(normalized_curvatures)[:, :3]
    
    # Assign colors based on the normalized average curvatures
    for i, color in enumerate(superpoint_features.keys()):
        indices = superpoint_features[color]['indices']
        colors[indices] = curvature_colors[i]
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # Visualize the point cloud colored by superpoint curvatures
    o3d.visualization.draw_geometries([pcd], window_name="Superpoint Curvature Visualization")


In [34]:
def color_and_visualize_region(pcd, superpoint_features, region, region_color=[0, 0, 0]):
    """Color all points in the final region the same and visualize the model."""
    colors = np.asarray(pcd.colors)
    
    # Set the color of all points in the region to the specified region color
    for superpoint_color in region:
        indices = superpoint_features[superpoint_color]['indices']
        colors[indices] = region_color
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # Visualize the updated point cloud
    o3d.visualization.draw_geometries([pcd], window_name="Region Growing Result")

In [53]:
def region_growing_superpoints(superpoint_features, pcd_tree, distance_threshold=0.5, threshold_normal=0.5, threshold_curvature=0.5):
    """Perform region growing on superpoints starting from a seed."""
    centroids = np.array([features['centroid'] for features in superpoint_features.values()])
    colors = list(superpoint_features.keys())
    
    # Precompute neighbors for each centroid
    neighbors = precompute_neighbors(pcd_tree, centroids, distance_threshold)
    print(neighbors)
    
    # Find the seed superpoint (example: highest point closest to centroid)
    seed_color = find_seed_superpoint(centroids, colors)
    
    visited = set()
    region = []
    stack = [seed_color]
    
    while stack:
        current_color = stack.pop()
        if current_color not in visited:
            visited.add(current_color)
            region.append(current_color)
            
            current_idx = colors.index(current_color)
            current_features = superpoint_features[current_color]
            
            for neighbor_idx in neighbors[current_idx]:
                neighbor_color = colors[neighbor_idx]
                neighbor_features = superpoint_features[neighbor_color]
                
                normal_diff = np.linalg.norm(current_features['avg_normal'] - neighbor_features['avg_normal'])
                curvature_diff = abs(current_features['avg_curvature'] - neighbor_features['avg_curvature'])
                
                if normal_diff < threshold_normal and curvature_diff < threshold_curvature:
                    stack.append(neighbor_color)
    
    return region


In [54]:
ply_file_path = 'Area_1_pbr28.ply'  # Replace with the path to your PLY file

# Load the PLY file with points, colors, normals, and curvatures
pcd, points, colors, normals, curvatures = load_ply_file(ply_file_path)

# Compute superpoint features based on unique colors
superpoint_features = compute_superpoint_features(points, colors, normals, curvatures)



In [48]:
#_, pcd.colors = find_seed_superpoint(np.array(pcd.points), np.array(pcd.colors))
o3d.visualization.draw_geometries([pcd], window_name="Region Growing Result")

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [12]:
# Visualize superpoints by average normals
visualize_superpoint_normals(pcd, superpoint_features)


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [18]:

# Visualize superpoints by average curvature
visualize_superpoint_curvature(pcd, superpoint_features)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [34]:
#visualize_curvature_colormap(pcd, curvatures)

In [55]:
# Create a KDTree for the centroids
centroids = np.array([features['centroid'] for features in superpoint_features.values()])
pcd_tree = KDTree(centroids)

In [56]:
# Perform region growing using the precomputed neighbors
region = region_growing_superpoints(superpoint_features, pcd_tree)

[array([ 2,  6,  8, 12, 14]), array([ 8, 16,  1]), array([2, 6, 8]), array([ 7,  3,  4, 15, 17, 18]), array([ 4, 10,  9, 15,  5, 17]), array([10,  9,  5]), array([ 2,  6,  8, 16, 14,  1,  7]), array([ 8, 16, 14, 11,  7,  3]), array([ 2,  6,  8, 16, 14, 11,  1,  7]), array([ 4, 10,  9, 15,  5]), array([ 4, 10,  9,  5]), array([16, 14, 11,  7]), array([12, 14, 13]), array([13, 10,  9]), array([ 6,  8, 16, 12, 14, 11,  7,  3]), array([ 4,  9, 15, 17, 18]), array([ 8, 16, 14, 11,  1,  7]), array([ 4, 15, 17, 18, 19]), array([15, 17, 18, 19]), array([18, 19])]


In [57]:
region

[(0.8941176470588236, 0.10196078431372549, 0.10980392156862745),
 (0.6, 0.6, 0.6),
 (1.0, 0.4980392156862745, 0.0),
 (0.6509803921568628, 0.33725490196078434, 0.1568627450980392),
 (0.596078431372549, 0.3058823529411765, 0.6392156862745098),
 (0.30196078431372547, 0.6862745098039216, 0.2901960784313726)]

In [ ]:
# Color all points in the final region and visualize the result
color_and_visualize_region(pcd, superpoint_features, region)